<a href="https://colab.research.google.com/github/EthanTong123/Quantum_Metal_designs/blob/main/design_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy==1.26.4
!pip install -q quantum-metal

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which 

ERROR: Operation cancelled by user
^C


In [ ]:
import qiskit_metal as qm
import matplotlib.pyplot as plt
from qiskit_metal import designs
from qiskit_metal.qlibrary.qubits.transmon_pocket_cl import TransmonPocketCL
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal import Dict

from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import (
    LaunchpadWirebondCoupled,
)
from collections import OrderedDict
import numpy as np

In [ ]:
design = designs.DesignPlanar()
design.overwrite_enabled = True
gui = qm.gui(design)

design.chips['main']['size']['size_x'] = '1mm'
design.chips['main']['size']['size_y'] = '1mm'

In [ ]:
options = dict(
    connection_pads = dict(
        a=dict(loc_W=+1, loc_H=+1),
        b=dict(loc_W=-1, loc_H=-1),
    ),
    cl_off_center="-25um",
)

q1 = TransmonPocketCL(design, "Q1", options=dict(cl_pocket_edge = 180, **options))
q2 = TransmonPocketCL(design, "Q2", options=dict(pos_x="-3mm", **options))

def connect(name, c1, p1, c2, p2, length, start_straight="0um", asym="0 um"):

    return RouteMeander(
        design,
        name,
        Dict(
            pin_inputs=Dict(
                start_pin=Dict(component=c1, pin=p1), end_pin=Dict(component=c2, pin=p2)
            ),
            lead=Dict(start_straight=start_straight),
            total_length=length,
            fillet="90um",
            meander=Dict(lead_start="0.1mm", lead_end="0.1mm", asymmetry=asym),
        ),
    )


cpw1 = connect("cpw1", "Q1", "b", "Q2", "a","10.0mm","0.15mm" ,"100um")

gui.autoscale()
gui.rebuild()

In [ ]:
# Wirebonds or sum - The corner ones, connects to chargeline
p1_c = LaunchpadWirebond(
    design,
    "P1_C",
    options=dict(pos_x="1mm", pos_y="-2.5mm", orientation="90", lead_length="0um"),
)
p2_c = LaunchpadWirebond(
    design,
    "P2_C",
    options=dict(pos_x="-4mm", pos_y="+2.5mm", orientation="270", lead_length="0um"),
)

# Exchange Coupler Lines to Edges
p1_q = LaunchpadWirebondCoupled(
    design,
    "P1_Q",
    options=dict(pos_x="1.45mm", pos_y="0", orientation="180", lead_length="30um"),
)
p2_q = LaunchpadWirebondCoupled(
    design,
    "P2_Q",
    options=dict(pos_x="-4.45mm", pos_y="0", orientation="0", lead_length="30um"),
)


gui.rebuild()

In [ ]:
# Connect Coupler Lines to Edges
ol1 = connect("ol1", "Q1", "a","P1_Q", "tie", "6mm", ".3mm","100um")
ol2 = connect("ol2", "Q2", "b","P2_Q", "tie", "6mm", ".3mm","100um")

gui.rebuild()

In [ ]:
#Chargelines to Corner
jogsB_in = OrderedDict()
# jogsB_in[0] = ["R", "0mm"]

# anchors = OrderedDict()
# anchors[0] = np.array([0.6, -2.5])
# anchors[1] = np.array([0.6, 0])

options_line_cl1 = {
    "pin_inputs": {
        "start_pin": {"component": "Q1", "pin": "Charge_Line"},
        "end_pin": {"component": "P1_C", "pin": "tie"},
    },
    "lead": {
        "start_straight": "0.25 mm",
        "end_straight": "0.1 mm",
    },
     "fillet" : "90 um"
}
cl1 = RouteAnchors(design, "line_cl1", options_line_cl1)

options_line_cl2 = {
    "pin_inputs": {
        "start_pin": {"component": "Q2", "pin": "Charge_Line"},
        "end_pin": {"component": "P2_C", "pin": "tie"},
    },
    "lead": {
        "start_straight": "0.25 mm",
        "end_straight": "0.1 mm",
    },
     "fillet" : "90 um"
}
cl2 = RouteAnchors(design, "line_cl2", options_line_cl2)

gui.rebuild()
gui.autoscale()

In [ ]:
#design.components['Q1'].options.connection_pads.keys()
#RouteAnchors.get_template_options(design)